In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Import Python libraries and locate the file
import pandas as pd
import numpy as np

file_path = "/content/drive/MyDrive/BMEN 600 Project/(GENE EXPRES.) GSE137765.txt"

In [ ]:
# Extract sample information
# Need to extract the sample IDs, tissue types, and contraceptive groups
# This creates a Pandas DataFrame called metadata, which contains information about each sample
import csv

sample_info = {}

with open(file_path, "r") as file:
    for line in file:
        if line.startswith("!Sample_"):
            values = next(csv.reader([line], delimiter="\t"))
            sample_info[values[0]] = values[1:]

sample_ids = sample_info["!Sample_geo_accession"]
sample_titles = sample_info["!Sample_title"]

metadata = pd.DataFrame({
    "sample_id": sample_ids,
    "sample_title": sample_titles
})

display(metadata.head())


,sample_id,sample_title
0,GSM4086934,cervical biopsy using control_Sample no: 3540
1,GSM4086935,cervical biopsy using LNG-IUD_Sample no: 3437
2,GSM4086936,endometrial biopsy using LNG-IUD_Sample no: 3265
3,GSM4086937,cervical biopsy using COC_Sample no: 3557
4,GSM4086938,cervical biopsy using LNG-IUD_Sample no: 3306


In [ ]:
# Categorize by contraceptive type
# This creates two new columns: contraceptive_group and tissue
def get_group(title):
    if "LNG-IUD" in title:
        return "LNG-IUD"
    elif "cu-IUD" in title:
        return "cu-IUD"
    elif "COC" in title:
        return "COC"
    elif "control" in title:
        return "control"
    else:
        return "unknown"

metadata["contraceptive_group"] = metadata["sample_title"].apply(get_group)

metadata["tissue"] = np.where(
    metadata["sample_title"].str.contains("endometrial", case=False),
    "endometrial",
    "cervical"
)

display(metadata[["sample_id", "tissue", "contraceptive_group"]].head(10))


,sample_id,tissue,contraceptive_group
0,GSM4086934,cervical,control
1,GSM4086935,cervical,LNG-IUD
2,GSM4086936,endometrial,LNG-IUD
3,GSM4086937,cervical,COC
4,GSM4086938,cervical,LNG-IUD
5,GSM4086939,endometrial,COC
6,GSM4086940,endometrial,LNG-IUD
7,GSM4086941,cervical,control
8,GSM4086942,cervical,cu-IUD
9,GSM4086943,cervical,cu-IUD


In [ ]:
# Count the samples
# Checks how many cervical and endometrial samples belong to each group
sample_counts = pd.crosstab(
    metadata["contraceptive_group"],
    metadata["tissue"]
)

display(sample_counts)


tissue,cervical,endometrial
contraceptive_group,,
COC,14,12
LNG-IUD,12,11
control,12,11
cu-IUD,14,13


In [ ]:
# Filtering/ remove cervical samples and cu-IUD samples
filtered_metadata = metadata[
    (metadata["tissue"] == "endometrial") &
    (metadata["contraceptive_group"] != "cu-IUD")
].copy()

display(filtered_metadata)

print("Remaining samples:", len(filtered_metadata))
print(filtered_metadata["contraceptive_group"].value_counts())


,sample_id,sample_title,contraceptive_group,tissue
2,GSM4086936,endometrial biopsy using LNG-IUD_Sample no: 3265,LNG-IUD,endometrial
5,GSM4086939,endometrial biopsy using COC_Sample no: 3485,COC,endometrial
6,GSM4086940,endometrial biopsy using LNG-IUD_Sample no: 3466,LNG-IUD,endometrial
10,GSM4086944,endometrial biopsy using COC_Sample no: 3174,COC,endometrial
11,GSM4086945,endometrial biopsy using LNG-IUD_Sample no: 3437,LNG-IUD,endometrial
14,GSM4086948,endometrial biopsy using control_Sample no: 3730,control,endometrial
21,GSM4086955,endometrial biopsy using control_Sample no: 3225,control,endometrial
22,GSM4086956,endometrial biopsy using COC_Sample no: 3557,COC,endometrial
29,GSM4086963,endometrial biopsy using control_Sample no: 3079,control,endometrial
35,GSM4086969,endometrial biopsy using COC_Sample no: 3612,COC,endometrial


Remaining samples: 34
contraceptive_group
COC        12
LNG-IUD    11
control    11
Name: count, dtype: int64


In [ ]:
# Load the gene expression data/ measurements
expression = pd.read_csv(
    file_path,
    sep="\t",
    skiprows=66,
    index_col=0,
    nrows=33297
)

print("Original expression matrix:", expression.shape)

display(expression.iloc[:5, :5])


Original expression matrix: (33297, 99)


,GSM4086934,GSM4086935,GSM4086936,GSM4086937,GSM4086938
ID_REF,,,,,
7892501,1.861056,1.755909,1.225046,1.551199,1.325084
7892502,3.425037,2.958231,5.583307,2.826053,4.022920
7892503,1.613959,2.548513,1.999543,1.672332,2.372853
7892504,8.187308,8.449696,8.903699,8.744138,8.693604
7892505,1.965319,2.328730,3.604877,3.421986,2.906380


In [ ]:
# Keep only the selected samples
# Original data remains unchanged, we have created filtered copies for the analysis
selected_samples = filtered_metadata["sample_id"].tolist()

expression_filtered = expression[selected_samples].copy()

print("Filtered expression matrix:", expression_filtered.shape)

display(expression_filtered.iloc[:5, :5])


Filtered expression matrix: (33297, 34)


,GSM4086936,GSM4086939,GSM4086940,GSM4086944,GSM4086945
ID_REF,,,,,
7892501,1.225046,2.510979,1.086197,2.068178,1.783205
7892502,5.583307,3.695185,2.904567,4.454361,3.072236
7892503,1.999543,1.708055,2.441292,3.624311,1.758640
7892504,8.903699,8.334480,9.306817,9.002178,8.154607
7892505,3.604877,1.662680,3.159033,1.252689,1.802251


In [ ]:
# Download the gene annotation table
import requests
from io import StringIO

url = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL6244&targ=self&form=text&view=full"

response = requests.get(url, timeout=120)
response.raise_for_status()

lines = response.text.splitlines()

start = lines.index("!platform_table_begin") + 1
end = lines.index("!platform_table_end")

annotation = pd.read_csv(
    StringIO("\n".join(lines[start:end])),
    sep="\t",
    low_memory=False
)

print(annotation.columns.tolist())
display(annotation.head())


['ID', 'GB_LIST', 'SPOT_ID', 'seqname', 'RANGE_GB', 'RANGE_STRAND', 'RANGE_START', 'RANGE_STOP', 'total_probes', 'gene_assignment', 'mrna_assignment', 'category']


,ID,GB_LIST,SPOT_ID,seqname,RANGE_GB,RANGE_STRAND,RANGE_START,RANGE_STOP,total_probes,gene_assignment,mrna_assignment,category
0,7896736,NaN,chr1:53049-54936,chr1,NC_000001.10,+,53049,54936,7,---,NONHSAT060105 // NONCODE // Non-coding transcr...,main
1,7896738,NaN,chr1:63015-63887,chr1,NC_000001.10,+,63015,63887,31,ENST00000328113 // OR4G2P // olfactory recepto...,ENST00000328113 // ENSEMBL // havana:known chr...,main
2,7896740,"NM_001004195,NM_001005240,NM_001005484,BC13684...",chr1:69091-70008,chr1,NC_000001.10,+,69091,70008,24,"NM_001004195 // OR4F4 // olfactory receptor, f...",NM_001004195 // RefSeq // Homo sapiens olfacto...,main
3,7896742,"NR_024437,XM_006711854,XM_006726377,XR_430662,...",chr1:334129-334296,chr1,NC_000001.10,+,334129,334296,6,NR_024437 // LOC728323 // uncharacterized LOC7...,NR_024437 // RefSeq // Homo sapiens uncharacte...,main
4,7896744,"NM_001005221,NM_001005224,NM_001005277,NM_0010...",chr1:367659-368597,chr1,NC_000001.10,+,367659,368597,36,"NM_001005221 // OR4F29 // olfactory receptor, ...",NM_001005221 // RefSeq // Homo sapiens olfacto...,main


In [ ]:
# Match gene names to expression values

# Find the column containing gene symbols
symbol_columns = [
    col for col in annotation.columns
    if "symbol" in col.lower()
]

print("Possible gene symbol columns:", symbol_columns)

Possible gene symbol columns: []


In [ ]:
print(annotation.columns.tolist())

['ID', 'GB_LIST', 'SPOT_ID', 'seqname', 'RANGE_GB', 'RANGE_STRAND', 'RANGE_START', 'RANGE_STOP', 'total_probes', 'gene_assignment', 'mrna_assignment', 'category']


In [ ]:
display(annotation.head())

,ID,GB_LIST,SPOT_ID,seqname,RANGE_GB,RANGE_STRAND,RANGE_START,RANGE_STOP,total_probes,gene_assignment,mrna_assignment,category
0,7896736,NaN,chr1:53049-54936,chr1,NC_000001.10,+,53049,54936,7,---,NONHSAT060105 // NONCODE // Non-coding transcr...,main
1,7896738,NaN,chr1:63015-63887,chr1,NC_000001.10,+,63015,63887,31,ENST00000328113 // OR4G2P // olfactory recepto...,ENST00000328113 // ENSEMBL // havana:known chr...,main
2,7896740,"NM_001004195,NM_001005240,NM_001005484,BC13684...",chr1:69091-70008,chr1,NC_000001.10,+,69091,70008,24,"NM_001004195 // OR4F4 // olfactory receptor, f...",NM_001004195 // RefSeq // Homo sapiens olfacto...,main
3,7896742,"NR_024437,XM_006711854,XM_006726377,XR_430662,...",chr1:334129-334296,chr1,NC_000001.10,+,334129,334296,6,NR_024437 // LOC728323 // uncharacterized LOC7...,NR_024437 // RefSeq // Homo sapiens uncharacte...,main
4,7896744,"NM_001005221,NM_001005224,NM_001005277,NM_0010...",chr1:367659-368597,chr1,NC_000001.10,+,367659,368597,36,"NM_001005221 // OR4F29 // olfactory receptor, ...",NM_001005221 // RefSeq // Homo sapiens olfacto...,main


In [ ]:
# Extract gene symbols from the gene_assignment column
annotation["gene_symbol"] = (
    annotation["gene_assignment"]
    .fillna("")
    .str.split(" // ")
    .str[1]
)

# Display the results
display(annotation[["ID", "gene_symbol"]].head(10))


,ID,gene_symbol
0,7896736,NaN
1,7896738,OR4G2P
2,7896740,OR4F4
3,7896742,LOC728323
4,7896744,OR4F29
5,7896746,MT-TM
6,7896748,MT-TW
7,7896750,MT-TD
8,7896752,MT-TK
9,7896754,LOC100287497


In [ ]:
# Match gene symbols to your expression data

# Make sure probe IDs have the same data type
annotation["ID"] = annotation["ID"].astype(str)
expression_filtered.index = expression_filtered.index.astype(str)

# Create a mapping between probe IDs and gene symbols
gene_mapping = annotation.set_index("ID")["gene_symbol"]

# Add gene symbols to the expression dataset
gene_expression = expression_filtered.copy()

gene_expression["gene_symbol"] = gene_expression.index.map(gene_mapping)

# Remove rows without a valid gene symbol
gene_expression = gene_expression.dropna(subset=["gene_symbol"])

gene_expression = gene_expression[
    gene_expression["gene_symbol"].str.strip().ne("")
    & gene_expression["gene_symbol"].ne("---")
]

display(gene_expression.head())


,GSM4086936,GSM4086939,GSM4086940,GSM4086944,GSM4086945,GSM4086948,GSM4086955,GSM4086956,GSM4086963,GSM4086969,...,GSM4087013,GSM4087014,GSM4087018,GSM4087019,GSM4087024,GSM4087026,GSM4087030,GSM4087031,GSM4087032,gene_symbol
ID_REF,,,,,,,,,,,,,,,,,,,,,
7896738,1.950945,1.907617,2.014378,2.346034,1.832857,1.894648,2.628919,1.762730,2.097927,1.801042,...,2.841932,2.159531,1.958396,1.782110,1.836963,2.010413,2.105290,2.158754,2.083437,OR4G2P
7896740,2.988975,2.168730,2.465069,2.870481,2.357692,2.624355,2.189264,1.814064,2.025361,2.164958,...,2.872860,2.061293,1.915194,2.123443,1.913821,2.308802,2.279870,2.666769,2.383967,OR4F4
7896742,7.072855,6.240329,5.820334,5.446926,6.129058,6.830575,5.700495,5.650570,5.673114,6.575070,...,4.308384,6.259729,5.700666,6.883442,5.675667,6.098445,6.865046,6.227093,6.122859,LOC728323
7896744,3.561565,3.812248,3.408147,3.291299,4.085403,4.817046,2.954132,3.517751,4.368667,3.801849,...,3.604017,3.197925,4.761172,3.495347,3.570542,3.394347,3.440071,3.227656,4.888315,OR4F29
7896746,10.180045,9.699657,10.151151,8.718475,8.422388,8.592460,10.135449,9.217355,9.393110,7.721013,...,10.082906,8.274448,8.947499,8.602974,8.546857,10.113178,10.146914,9.345997,8.954704,MT-TM


In [ ]:
# Organize the data by gene name
# Group rows by gene symbol
gene_expression = gene_expression.groupby("gene_symbol").mean()

# Display the first five genes
display(gene_expression.head())


,GSM4086936,GSM4086939,GSM4086940,GSM4086944,GSM4086945,GSM4086948,GSM4086955,GSM4086956,GSM4086963,GSM4086969,...,GSM4087012,GSM4087013,GSM4087014,GSM4087018,GSM4087019,GSM4087024,GSM4087026,GSM4087030,GSM4087031,GSM4087032
gene_symbol,,,,,,,,,,,,,,,,,,,,,
A1BG,4.908607,4.962685,5.190040,5.239006,4.876340,5.001215,4.982528,5.245752,5.376415,5.195965,...,5.032040,5.625346,5.102428,5.311457,5.328728,4.789494,5.024219,5.161185,5.312145,5.037547
A1CF,2.843271,2.683095,2.520372,3.380781,2.848569,2.834005,2.961477,3.218246,2.940084,3.037475,...,2.936910,3.189573,2.720271,2.802437,3.068765,3.081822,3.040880,3.199125,3.315057,3.200907
A2M,9.043931,6.547989,9.233096,7.677573,9.025902,9.043810,7.961733,8.926874,8.708647,8.741877,...,7.839787,9.130802,8.842600,9.044788,9.356286,9.012527,8.018010,7.617431,8.669140,8.981040
A2ML1,4.282296,4.322072,3.749713,3.805974,3.508045,4.522006,3.784617,3.408761,3.990611,3.577150,...,4.879323,4.532624,3.404791,3.822905,3.775249,4.295873,4.799154,4.591019,3.945224,3.875684
A3GALT2,2.824455,2.528947,2.998205,3.949480,3.186370,3.231141,3.502371,3.096704,3.175226,3.096743,...,3.685634,3.278680,3.372299,3.126388,2.703197,3.415039,3.625614,3.153767,3.381258,3.528806


In [ ]:
# Check whether the six ECM genes we selected are present
ecm_genes = ["EPYC", "MMP7", "MMP12", "MMP26", "ITGA6", "ITGAM"]

for gene in ecm_genes:
    if gene in gene_expression.index:
        print(gene, "FOUND")
    else:
        print(gene, "NOT FOUND")


EPYC FOUND
MMP7 FOUND
MMP12 FOUND
MMP26 FOUND
ITGA6 FOUND
ITGAM FOUND


# Creating the table for our selected 6 genes

In [ ]:
# Creating the 7x4 table
ecm_table = pd.DataFrame(index=ecm_genes)

for group in ["LNG-IUD", "COC", "control"]:

    group_ids = filtered_metadata.loc[
        filtered_metadata["contraceptive_group"] == group,
        "sample_id"
    ]

    ecm_table[group] = gene_expression.reindex(ecm_genes)[group_ids].mean(axis=1)

ecm_table.index.name = "Gene"

display(ecm_table.round(3))

# Values are the log2 expression of that gene within the assigned contraceptive group
# The higher the value, the more expressed the gene is


,LNG-IUD,COC,control
Gene,,,
EPYC,8.238,3.787,3.073
MMP7,9.441,8.365,5.916
MMP12,5.973,2.706,2.660
MMP26,2.767,3.730,5.547
ITGA6,7.304,7.237,8.256
ITGAM,6.916,5.273,4.889


In [ ]:
# Save the 7x4 ECM gene expression table to Google Drive

ecm_table.to_csv(
    "/content/drive/MyDrive/BMEN 600 Project/ECM_Gene_Expression_Table.csv"
)

print("7x4 ECM gene expression table saved to Google Drive!")


7x4 ECM gene expression table saved to Google Drive!


# Generating the top 10 differentially expressed genes

Table 1: Top 10 differentially expressed genes: LNG-IUD vs control

Table 2: Top 10 differentially expressed genes: COC vs control

In [ ]:
# Calculate differential expression

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np

def differential_expression(group_name):

    # Get sample IDs for the contraceptive group
    group_ids = filtered_metadata.loc[
        filtered_metadata["contraceptive_group"] == group_name,
        "sample_id"
    ]

    # Get sample IDs for the control group
    control_ids = filtered_metadata.loc[
        filtered_metadata["contraceptive_group"] == "control",
        "sample_id"
    ]

    # Extract gene expression values
    group_values = gene_expression[group_ids]
    control_values = gene_expression[control_ids]

    # Calculate log2 fold change
    log2_fc = group_values.mean(axis=1) - control_values.mean(axis=1)

    # Perform Welch's t-test
    _, p_values = ttest_ind(
        group_values,
        control_values,
        axis=1,
        equal_var=False,
        nan_policy="omit"
    )

    # Create results table
    results = pd.DataFrame({
        "Gene": gene_expression.index,
        "log2_fold_change": log2_fc.values,
        "p_value": p_values
    })

    # Apply FDR correction
    valid = results["p_value"].notna()
    results["FDR"] = np.nan

    results.loc[valid, "FDR"] = multipletests(
        results.loc[valid, "p_value"],
        method="fdr_bh"
    )[1]

    return results.sort_values("FDR")

In [ ]:
# Generate the top 10 genes for LNG-IUD
lng_results = differential_expression("LNG-IUD")

lng_top10 = lng_results[
    lng_results["FDR"] <= 0.05
].head(10)

display(lng_top10)

# Gene - gene symbol
# log2_fold_change - difference in mean log2 expression relative to control
# p_value - unadjusted statistical p-value
# FDR - P-value adjusted for multiple testing

,Gene,log2_fold_change,p_value,FDR
2163,C6orf141,-2.116740,1.192345e-09,0.000028
3764,CRISP3,-3.992445,2.422178e-09,0.000028
2672,CCL2,3.288327,6.626073e-09,0.000051
10192,MANSC1,-1.466673,2.532224e-08,0.000061
4571,DNAJC15,-1.124605,2.071194e-08,0.000061
11191,MT1H,-3.273813,2.064945e-08,0.000061
3972,CWH43,-2.382136,2.447804e-08,0.000061
11188,MT1E,-1.901278,1.553576e-08,0.000061
11189,MT1F,-2.815704,1.287779e-08,0.000061
11190,MT1G,-2.962858,2.625241e-08,0.000061


In [ ]:
# Generate the top 10 genes for COC
coc_results = differential_expression("COC")

coc_top10 = coc_results[
    coc_results["FDR"] <= 0.05
].head(10)

display(coc_top10)

# Gene - gene symbol
# log2_fold_change - difference in mean log2 expression relative to control
# p_value - unadjusted statistical p-value
# FDR - P-value adjusted for multiple testing

,Gene,log2_fold_change,p_value,FDR
11196,MT1M,-3.040453,2.225691e-08,0.000519
11195,MT1L,-2.769434,1.715835e-07,0.001481
11199,MT1X,-2.153847,1.906903e-07,0.001481
11189,MT1F,-2.440787,3.612304e-07,0.001809
11190,MT1G,-2.879397,3.880735e-07,0.001809
11197,MT1P1,-2.291937,4.666768e-07,0.001813
11201,MT2P1,-1.813904,5.758338e-07,0.001917
1316,ATRNL1,1.500083,6.800660e-07,0.001981
10556,MGC24103,1.604624,1.954135e-06,0.005061
11200,MT2A,-1.446069,2.388781e-06,0.005442


In [ ]:
# Saving the tables in my drive
folder = "/content/drive/MyDrive/BMEN 600 Project/"

lng_top10.to_csv(folder + "Top10_LNG_IUD.csv", index=False)
coc_top10.to_csv(folder + "Top10_COC.csv", index=False)

print("Both tables saved to Google Drive!")

Both tables saved to Google Drive!
